In [1]:
import numpy as np
from genriesz import grr_ame, SquaredGenerator, PolynomialBasis

rng = np.random.default_rng(0)


In [10]:
n = 4000
d = 3

X = rng.multivariate_normal([0, 1], [[1, 0.5], [0.5, 1]], size=n)
eps = rng.normal(scale=1.0, size=n)

Y = 1 + np.sin(X[:, 0]) + 0.5 * (X[:, 1] ** 2) + eps

true_ame0 = float(np.exp(-0.5))  # E[cos(N(0,1))]
print("Approx. true AME for coordinate 0:", true_ame0)

Approx. true AME for coordinate 0: 0.6065306597126334


In [11]:
# A simple polynomial basis on X
basis = PolynomialBasis(degree=3, include_bias=True)

gen = SquaredGenerator(C=0.0).as_generator()

res = grr_ame(
    X=X,
    Y=Y,
    coordinate=0,
    basis=basis,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res.summary_text())



AME(coord=0) estimates (n=4000)
alpha=0.05 | null=0.0

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                0.613418    0.00732371       [ 0.599064,  0.627772]           0
RW                 0.60936     0.0525976        [ 0.506271,  0.71245]           0
ARW               0.619343     0.0203833       [ 0.579393,  0.659294]           0
TMLE              0.619233      0.020418       [ 0.579214,  0.659251]           0


In [12]:
from genriesz import UKLGenerator, BPGenerator

# A small grid over generators and regularization.
generator_grid = [
    ("SQ", SquaredGenerator(C=0.0).as_generator()),
    ("UKL (C=0)", UKLGenerator(C=0.0).as_generator()),
    ("BP (omega=0.1, C=0)", BPGenerator(C=0.0, omega=0.1).as_generator()),
    ("BP (omega=0.2, C=0)", BPGenerator(C=0.0, omega=0.2).as_generator()),
    ("BP (omega=0.5, C=0)", BPGenerator(C=0.0, omega=0.5).as_generator()),
]

penalty_grid = [
    {"penalty": "l2", "lam": 1e-4},
    {"penalty": "l2", "lam": 1e-3},
    {"penalty": "l1", "lam": 1e-4},
    {"penalty": "l1", "lam": 1e-3},
]

rows = []
for gname, gen_i in generator_grid:
    for cfg in penalty_grid:
        res_i = grr_ame(
            X=X,
            Y=Y,
            coordinate=0,
            basis=basis,
            generator=gen_i,
            cross_fit=True,
            folds=3,               # smaller folds for the sweep
            random_state=0,
            estimators=("ra", "rw", "arw", "tmle"),
            outcome_models="shared",
            outcome_link="identity",  # Y is unbounded, so Gaussian TMLE is appropriate
            riesz_penalty=cfg["penalty"],
            riesz_lam=cfg["lam"],
            max_iter=250,
            tol=1e-8,
        )

        row = {
            "generator": gname,
            "penalty": cfg["penalty"],
            "lam": cfg["lam"],
        }

        for k in ("ra", "rw", "arw", "tmle"):
            e = res_i.estimates[k]
            row[f"{k}"] = e.estimate
            row[f"{k}_se"] = e.se
            row[f"{k}_err"] = e.estimate - true_ame0

        rows.append(row)

import pandas as pd

df = pd.DataFrame(rows)
# Sort by absolute ARW error (ARW is typically stable)
df = df.sort_values(by="arw_err", key=lambda s: np.abs(s))
display(df)

,generator,penalty,lam,ra,ra_se,ra_err,rw,rw_se,rw_err,arw,arw_se,arw_err,tmle,tmle_se,tmle_err
9,"BP (omega=0.1, C=0)",l2,0.0010,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
17,"BP (omega=0.5, C=0)",l2,0.0010,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
16,"BP (omega=0.5, C=0)",l2,0.0001,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
15,"BP (omega=0.2, C=0)",l1,0.0010,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
14,"BP (omega=0.2, C=0)",l1,0.0001,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
13,"BP (omega=0.2, C=0)",l2,0.0010,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
12,"BP (omega=0.2, C=0)",l2,0.0001,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
11,"BP (omega=0.1, C=0)",l1,0.0010,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
10,"BP (omega=0.1, C=0)",l1,0.0001,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
19,"BP (omega=0.5, C=0)",l1,0.0010,0.613022,0.007392,0.006491,2.035650,0.030364,1.429119,0.612400,0.017557,0.005869,0.613022,0.017557,0.006491
